# Fourier Analysis and Guided Initialization

Refactored notebook to study the periodic VQE landscape without noise. The goal is to show when a low-order Fourier approximation helps choose a better initial point for the optimizer.


## Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.fourier import (
    analyze_fourier_line,
    build_fourier_problem,
    estimate_first_harmonic_guided_point,
    make_coordinate_direction,
    run_budget_comparison,
    run_vqe_reference_point,
    scan_harmonic_error,
    scan_spectral_profile,
    spectral_metrics,
)
from src.vqe.ansatz import build_ansatz
from src.vqe.molecular_system import default_statevector_systems
from src.visualization.fourier_plots import (
    plot_budget_comparison,
    plot_fourier_reconstruction,
    plot_harmonic_error,
    plot_harmonic_profile,
    plot_spectral_metrics,
    save_figure,
)

pd.set_option("display.max_columns", None)

output_dir = "outputs/figures/fourier"
seed = 137


## Systems and Configuration

We start with `H2`, `LiH`, and `BeH2` in `sto-3g`, using a small distance grid. This keeps the notebook practical for generating slide figures.


In [ ]:
systems = [
    system
    for system in default_statevector_systems()
    if system.name in {"H2", "LiH", "BeH2"}
]

ansatz_name = "real_amplitudes"
reps = 2
mapper = "parity"
z2symmetry_reduction = True
optimizer_name = "cobyla"
reference_max_iter = 250
theta_samples = 64

[(system.name, system.basis, system.distances, system.active_space) for system in systems]


## Demonstration: Fourier Lines for H2, LiH, and BeH2

For each molecule, we choose a point near the VQE optimum, vary one parameter at a time, and reconstruct `E(theta)` with a few harmonics.


In [ ]:
fourier_lines = {}
fourier_contexts = {}

for demo_system in systems:
    demo_distance = demo_system.distances[len(demo_system.distances) // 2]

    problem, qubit_op, constant_energy = build_fourier_problem(
        demo_system,
        demo_distance,
        mapper=mapper,
        z2symmetry_reduction=z2symmetry_reduction,
    )
    ansatz = build_ansatz(
        name=ansatz_name,
        num_qubits=qubit_op.num_qubits,
        reps=reps,
        num_particles=problem.num_particles,
        num_spatial_orbitals=problem.num_spatial_orbitals,
    )

    vqe_reference = run_vqe_reference_point(
        qubit_op=qubit_op,
        ansatz=ansatz,
        constant_energy=constant_energy,
        optimizer_name=optimizer_name,
        max_iter=reference_max_iter,
        seed=seed,
    )
    if not vqe_reference.get("success", False):
        print(f"Falha no VQE de referência para {demo_system.name}: {vqe_reference.get('error')}")
        continue

    center = np.asarray(vqe_reference["optimal_params"], dtype=float)
    direction = make_coordinate_direction(ansatz.num_parameters, parameter_index=0)
    line = analyze_fourier_line(
        ansatz=ansatz,
        qubit_op=qubit_op,
        constant_energy=constant_energy,
        center=center,
        direction=direction,
        theta_samples=theta_samples,
    )

    fourier_lines[demo_system.name] = line
    fourier_contexts[demo_system.name] = {
        "system": demo_system,
        "distance": demo_distance,
        "problem": problem,
        "qubit_op": qubit_op,
        "constant_energy": constant_energy,
        "ansatz": ansatz,
        "center": center,
        "direction": direction,
    }

pd.DataFrame([
    {
        "molecule": molecule,
        "basis": ctx["system"].basis,
        "distance": ctx["distance"],
        **spectral_metrics(line.coefficients),
    }
    for molecule, line in fourier_lines.items()
    for ctx in [fourier_contexts[molecule]]
])


In [ ]:
fig, axes = plt.subplots(1, len(fourier_lines), figsize=(6 * len(fourier_lines), 4), squeeze=False)

for ax, (molecule, line) in zip(axes.ravel(), fourier_lines.items()):
    plot_fourier_reconstruction(line, harmonic_orders=(1, 2, 3), ax=ax)
    ctx = fourier_contexts[molecule]
    ax.set_title(f"{molecule} ({ctx['basis'] if 'basis' in ctx else ctx['system'].basis}, d={ctx['distance']:.3f} Å)")

path = save_figure(fig, output_dir, "fourier_reconstruction_H2_LiH_BeH2.png")
print(path)
plt.show()


## Guided Initial Point

Using three evaluations at `theta = 0, pi/2, -pi/2`, we estimate the first harmonic and compute the analytical minimum of that approximation.


In [ ]:
demo_name = "H2"
ctx = fourier_contexts[demo_name]

guided_point, guide_cost, guide_info = estimate_first_harmonic_guided_point(
    ansatz=ctx["ansatz"],
    qubit_op=ctx["qubit_op"],
    constant_energy=ctx["constant_energy"],
    center=ctx["center"],
    direction=ctx["direction"],
)

pd.DataFrame([{
    "molecule": demo_name,
    "basis": ctx["system"].basis,
    "distance": ctx["distance"],
    "guide_cost": guide_cost,
    **guide_info,
}])


## Local vs Global Spectral Profile

`R1` measures how much spectral weight is concentrated in the first harmonic. `H_norm` measures how spread out the energy is across harmonics.


In [ ]:
RUN_SPECTRAL_SCAN = True

if RUN_SPECTRAL_SCAN:
    spectral_df, profile_df = scan_spectral_profile(
        systems=systems,
        ansatz_name=ansatz_name,
        reps=reps,
        mapper=mapper,
        z2symmetry_reduction=z2symmetry_reduction,
        optimizer_name=optimizer_name,
        max_iter=reference_max_iter,
        theta_samples=theta_samples,
        seed=seed,
        global_samples=3,
        max_harmonics=8,
    )

    display(spectral_df.head())
    display(profile_df.head())


In [ ]:
if RUN_SPECTRAL_SCAN:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_harmonic_profile(profile_df, ax=ax)
    path = save_figure(fig, output_dir, "fourier_harmonic_profile.png")
    print(path)
    plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    plot_spectral_metrics(spectral_df, ax=ax)
    path = save_figure(fig, output_dir, "fourier_spectral_metrics.png")
    print(path)
    plt.show()


## Error vs Harmonic Order

This block shows whether `K=1` is already enough or whether additional harmonics are needed to reconstruct the curve and its minimum.


In [ ]:
RUN_HARMONIC_ERROR = True

if RUN_HARMONIC_ERROR:
    harmonic_error_df = scan_harmonic_error(
        systems=systems,
        harmonic_grid=(1, 2, 3, 5),
        ansatz_name=ansatz_name,
        reps=reps,
        mapper=mapper,
        z2symmetry_reduction=z2symmetry_reduction,
        optimizer_name=optimizer_name,
        max_iter=reference_max_iter,
        theta_samples=theta_samples,
        seed=seed,
    )

    harmonic_summary = (
        harmonic_error_df.groupby(["molecule", "K"], as_index=False)
        .agg(mean_rmse=("rmse", "mean"), mean_delta_min=("delta_min_energy", "mean"))
        .sort_values(["molecule", "K"])
    )
    display(harmonic_summary)


In [ ]:
if RUN_HARMONIC_ERROR:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    plot_harmonic_error(harmonic_error_df, ax=ax)
    path = save_figure(fig, output_dir, "fourier_error_vs_k.png")
    print(path)
    plt.show()


## Random Initialization vs Fourier-Guided

Main hypothesis: guided initialization improves the same random initial point, especially when the optimizer has a small iteration budget.


In [ ]:
RUN_BUDGET_COMPARISON = True

if RUN_BUDGET_COMPARISON:
    budget_df = run_budget_comparison(
        systems=systems,
        iteration_grid=(25, 50, 100, 200),
        ansatz_name=ansatz_name,
        reps=reps,
        mapper=mapper,
        z2symmetry_reduction=z2symmetry_reduction,
        optimizer_name=optimizer_name,
        seed=seed,
        repeats=3,
    )

    budget_summary = (
        budget_df.groupby(["max_iter", "molecule", "mode"], as_index=False)
        .agg(mean_total_cost=("total_cost", "mean"), mean_abs_error=("abs_error", "mean"))
        .sort_values(["molecule", "max_iter", "mode"])
    )
    display(budget_summary)


In [ ]:
if RUN_BUDGET_COMPARISON:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    plot_budget_comparison(budget_df, ax=ax)
    path = save_figure(fig, output_dir, "fourier_budget_comparison.png")
    print(path)
    plt.show()


## Data for Slides


In [ ]:
for name in ["spectral_df", "profile_df", "harmonic_error_df", "budget_df"]:
    if name in globals():
        print(name, globals()[name].shape)

# Exemplo de tabela compacta para copiar para slide/texto.
if "budget_df" in globals():
    display(
        budget_df.groupby(["max_iter", "molecule", "mode"], as_index=False)
        .agg(
            mean_error=("abs_error", "mean"),
            median_error=("abs_error", "median"),
            mean_total_cost=("total_cost", "mean"),
        )
        .sort_values(["molecule", "max_iter", "mode"])
    )


### Initialization Relevance vs Budget
This cell summarizes the central idea: guided initialization tends to matter more when the optimizer has few iterations.
With more iterations, the difference between starting from a random point and from the Fourier-guided point should shrink.


In [ ]:
if "budget_df" not in globals():
    raise RuntimeError("Execute primeiro a seção 'Inicialização Aleatória vs Fourier-Guided'.")

iteration_effect = (
    budget_df.groupby(["max_iter", "molecule", "mode"], as_index=False)
    .agg(
        mean_abs_error=("abs_error", "mean"),
        median_abs_error=("abs_error", "median"),
        mean_total_cost=("total_cost", "mean"),
    )
)

iteration_wide = iteration_effect.pivot_table(
    index=["max_iter", "molecule"],
    columns="mode",
    values=["mean_abs_error", "median_abs_error", "mean_total_cost"],
    aggfunc="first",
).reset_index()
iteration_wide.columns = [
    "max_iter",
    "molecule",
    "mean_error_guided",
    "mean_error_random",
    "median_error_guided",
    "median_error_random",
    "mean_cost_guided",
    "mean_cost_random",
]

iteration_wide["absolute_error_gain"] = (
    iteration_wide["mean_error_random"] - iteration_wide["mean_error_guided"]
)
iteration_wide["relative_error_gain"] = 1.0 - (
    iteration_wide["mean_error_guided"] / (iteration_wide["mean_error_random"] + 1e-16)
)
iteration_wide["cost_delta"] = (
    iteration_wide["mean_cost_guided"] - iteration_wide["mean_cost_random"]
)

display(iteration_wide.sort_values(["molecule", "max_iter"]).round(8))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for molecule, group in iteration_wide.groupby("molecule"):
    group = group.sort_values("max_iter")
    ax[0].plot(
        group["max_iter"],
        group["absolute_error_gain"],
        marker="o",
        linewidth=2,
        label=molecule,
    )
    ax[1].plot(
        group["max_iter"],
        group["relative_error_gain"],
        marker="o",
        linewidth=2,
        label=molecule,
    )

ax[0].axhline(0.0, color="#7a003c", linewidth=1)
ax[0].set_title("Ganho absoluto do Fourier-guided")
ax[0].set_xlabel("Máximo de iterações")
ax[0].set_ylabel("erro_random - erro_guided")
ax[0].grid(alpha=0.25)

ax[1].axhline(0.0, color="#7a003c", linewidth=1)
ax[1].set_title("Ganho relativo do Fourier-guided")
ax[1].set_xlabel("Máximo de iterações")
ax[1].set_ylabel("1 - erro_guided / erro_random")
ax[1].grid(alpha=0.25)
ax[1].legend()

path = save_figure(fig, output_dir, "fourier_initialization_relevance_vs_iterations.png")
print(path)
plt.show()
